# Model check: same excitation, simulator vs models

Drives the MuJoCo simulator (`make_calibrated_sim`, treated as the real robot) and five models
with **the same commands**, then compares the tip per axis:

| Model | Built by | Inputs it takes | Outputs |
|---|---|---|---|
| One-mode model | Nikoletta (latest notebook, Smith-2 §7): ÿ + 9.231 ẏ + 10.60 y = b Δp | plane pressure differences: x ← −u₃, y ← u₂ − u₄ (no actuator dynamics) | x, y |
| One-mode + gravity | Nikoletta's derivation with the chain's gravity stiffness added (our review, not in her notebooks): k = 51.45 | same | x, y |
| M2 | Ibraheem, `task_space_model.py`: first-order mechanics | the three segment commands, through dead time 0.5 s + lag 0.6 s | x, y, z |
| M3 | Ibraheem, `task_space_model.py`: M2's actuator + second-order mechanics (inertia) | same as M2 | x, y, z |
| F | Ibraheem, `sysid_frf.py` | channels ν_x = −u₃, ν_y = u₂ − u₄, with the identified delay | x, y |

u = command − 4.5 psi. Every model runs **open loop from rest** (free run): it never sees the
simulator's tip, so errors accumulate exactly as they would in a controller's internal model.

**Plots.** One overview with every model, then one figure per model against the simulator
(thick black line). Each model keeps its own colour, line style and marker throughout.

All code is in `scripts/compare_models_excitation.py`; this notebook only calls it and plots.
Runs offline in the project environment (`uv run jupyter lab`).

## 0 — What S2, S3, S4 are, and what an excitation is

The arm has four pressure columns around it, S1–S4. Inflating one bends the tip toward its side:

| Column | pushes the tip toward | on `make_calibrated_sim()` |
|---|---|---|
| S1 | +x | sealed reservoir, not commandable |
| S2 | +y | input u₂ |
| S3 | −x | input u₃ |
| S4 | −y | input u₄ |

So the robot has **three inputs**, the commanded pressures u = (u₂, u₃, u₄) in psi, and every
controller outputs exactly these three numbers each step (`docs/simulator.md`).

An **excitation** is one test input: a pressure trace for each of the three inputs, played at the
same time. The three traces are different on purpose: S2 and S4 push in opposite directions, so
equal pressures cancel in bending and would only test the z lift. Independent traces move the tip
in every direction and test every input combination.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
import compare_models_excitation as CM
import plot_style as P

P.apply()
%matplotlib inline

## 1 — Choose an excitation

- `"random_steps"`: independent random levels per segment, 1.5–7.5 psi, held 0.5–3 s.
- `"chirp"`: phased sines (0°, 120°, 240°) on S2–S4, 4.5 ± 2.5 psi, frequency swept 0.02 → 1 Hz.
- Or set `CMDS` yourself: any `(n, 3)` array of absolute psi for S2, S3, S4 at 100 Hz, within 0–9.

Seeds used for identification and validation elsewhere: 1–4, 11–14, 21–22, 101. Pick another.

In [ ]:
KIND = "random_steps"     # or "chirp"
SECONDS = 60.0
SEED = 31                 # random_steps only

CMDS = CM.excitation(KIND, SECONDS, seed=SEED)
# CMDS = np.column_stack([...])   # custom excitation: (n, 3) psi, 100 Hz

t = np.arange(len(CMDS)) * CM.DT
fig, ax = plt.subplots(figsize=(P.WIDTH, 3.5))
for j, seg in enumerate(("S2", "S3", "S4")):
    ax.plot(t, CMDS[:, j], **P.line(seg, len(t)))
ax.set(xlabel="time (s)", ylabel="command (psi)", title=f"Excitation: {KIND}")
ax.grid(alpha=0.3); P.legend_outside(ax)
plt.show()

## 1b — The actuator model from the ground up

**What it describes.** The command u is a pressure set-point sent to a valve. The pressure p
inside that column's pouches is what bends the arm. The actuator model is the map u → p.

**Dead time T.** A new command does not reach the pouch for T seconds (transport through the
valve and the line). During that time p does not change at all:  p(t) depends on u(t − T).

**Lag τ.** The pouch then fills through a flow restriction. Linearised, the flow is proportional
to the pressure difference, and pressure rises in proportion to the flow:
C ṗ = (G u(t − T) − p) / R, with R the pneumatic resistance and C the pouch's capacitance. With τ = R C:

  **τ ṗ = −p + G u(t − T)**      transfer function  P(s) / U(s) = G e^(−sT) / (τ s + 1)

- p: pouch pressure above its settled value (psi); u: command above 4.5 psi (psi)
- G: static gain (psi of pouch pressure per psi commanded); τ: time constant (s); T: dead time (s)

So yes: one linear first-order ODE plus a pure delay, one independent copy per input (three in
total, no coupling between them). The simulator implements exactly this form
(`p_actual += (dt / tau) (p_target − p_actual)` behind a delay buffer, `docs/simulator.md`).

**Step response.** For a step Δu at t = 0:  p(t) = 0 for t < T, and
p(t) = G Δu (1 − e^(−(t − T)/τ)) for t ≥ T. Read off: T = when p first moves,
τ = time from T to 63 % of the final change, G = final change / Δu.

**Identifying T, τ, G without knowing them** (the cell below does the first two on the simulator):
1. Step one input, record the measured pouch pressure: T = first sample outside the noise band.
2. Fit the step response above by least squares for T, τ, G.
3. With arbitrary data instead of a step: for each candidate delay N, fit p[k+1] = α p[k] + β u[k − N]
   by least squares, keep the N with the smallest free-run error; τ = −Δt / ln α, G = β / (1 − α).
4. Frequency response: the phase is −ωT − arctan(ωτ), so the phase's slope at high ω gives T.
5. Without a pressure sensor, the same fits on the tip still find T, but τ mixes with the
   mechanics' own time constant; measuring the pouch pressure separates them.

In [ ]:
act = CM.identify_actuator(seg=1)      # +1 psi step on S3 after 7 s at 4.5 psi
print(f"noise std {act['noise_std']:.4f} psi;  first rise at {act['dead_time_first_rise_s']:.2f} s")
print(f"least-squares fit:  T = {act['dead_time_s']:.3f} s,  tau = {act['tau_s']:.3f} s,  G = {act['gain']:.3f} psi/psi")
CM.figure_actuator(act)
plt.show()

## 2 — Run the simulator and the five models

In [ ]:
u = CMDS - CM.BASE
truth = CM.simulate_truth(CMDS)                  # MuJoCo, seed 0, 5 s settle at 4.5 psi first
sims = {m: CM.SIMULATE[m](u) for m in CM.MODELS}
results = {m: CM.metrics(truth, sims[m]) for m in CM.MODELS}

print(f"{'model':<30}" + "".join(f"{a + ' RMSE (fit)':>22}" for a in "xyz"))
for m in CM.MODELS:
    cells = [f"{v['rmse_mm']:.3f} mm ({v['fit_pct']:5.1f} %)" if v else "no z output" for v in results[m].values()]
    print(f"{CM.LABELS[m]:<30}" + "".join(f"{c:>22}" for c in cells))

**Reading the numbers.** RMSE is in mm. Fit = 100 (1 − ‖model − truth‖ / ‖truth − mean‖): 100 %
is perfect, 0 % is as good as predicting a constant, negative is worse than a constant.

## 3 — Tip per axis: all models, then each model against the simulator

In [ ]:
CM.figure_overview(KIND, t, CMDS, truth, sims)
plt.show()
for m in CM.MODELS:
    CM.figure_model(KIND, m, t, truth, sims, results)
    plt.show()

## 4 — Error over time, M2, M3 and F (model − simulator)

The one-mode models' errors are 30–100× larger and would hide these, so they are left out here.

In [ ]:
fig, axs = P.figure(3, 1, sharex=True, layout="constrained")
for i, name in enumerate("xyz"):
    ax = axs[i, 0]
    for m in ("M2", "M3", "F"):
        if not np.isnan(sims[m][:, i]).all():
            ax.plot(t, sims[m][:, i] - truth[:, i], **P.line(CM.LABELS[m], len(t)))
    ax.axhline(0, color="black", lw=1)
    ax.set(ylabel=f"{name} error (mm)")
    ax.grid(alpha=0.3)
P.legend_outside(axs[0, 0])
axs[-1, 0].set_xlabel("time (s)")
plt.show()

## 5 — Save the data

Writes the commands, the simulator's tip and every model's tip (mm, 100 Hz) to an `.npz` file,
and the metrics to a `.json` file next to it. Load later with `np.load(path)`.

In [ ]:
import json
out = ROOT / "docs" / "figures" / f"model_check_{KIND}_seed{SEED}"
np.savez_compressed(out.with_suffix(".npz"), t=t, cmds=CMDS, truth=truth, **{m: s for m, s in sims.items()})
out.with_suffix(".json").write_text(json.dumps(dict(kind=KIND, seconds=SECONDS, seed=SEED, metrics=results), indent=1))
print("saved", out.with_suffix(".npz"), "and", out.with_suffix(".json"))

## 6 — M2 + nonlinear disturbance observer (NDO)

An NDO is not a model on its own: it estimates the part of the motion the model does not explain
**from the measured tip**. In a free run there is no measurement, so it would add nothing. It is
therefore scored on the job the delay-aware controllers give the model: predicting the tip one
dead time T = 0.5 s ahead from the tip measured up to now (the Smith predictor).

- hold:  ŷ(t+T) = y(t)  (no model; what an undelayed controller implicitly assumes)
- M2:  ŷ(t+T) = y(t) + x_m(t) − x_m(t−T), with x_m = M2 without the dead time, driven by the commands
  already sent: the commands of the last T have not reached the tip yet, and M2 says how far they will move it
- M2 + NDO:  M2 + d̂ (1 − e^(−aT)) / a, where d̂ (mm/s) is the observer's estimate of the
  unmodelled rate in ẏ = −a y + a H p + d, held constant over T. Observer as in `ndosmc.NDOB`
  (l₂ = 0): d̂ = z + l₁ y,  ż = l₁ (a y − a H p̂ − d̂), i.e. d̂ is the model's rate residual
  low-passed at l₁ rad/s. l₁ = 0.1 is the gain used in the delay-aware NDOSMC.

In [ ]:
nd = CM.m2_parts()[4]
print(f"{'predictor (T = ' + str(nd * CM.DT) + ' s)':<28}" + "".join(f"{a + ' RMSE (mm)':>16}" for a in "xyz"))
for name, v in CM.prediction_metrics(truth, u).items():
    print(f"{name:<28}" + "".join(f"{e:>16.3f}" for e in v.values()))

CM.figure_prediction(t, truth, u)
plt.show()

**Result (both standard excitations, 60 s).** The NDO does not improve M2's prediction; it makes it
slightly worse (random steps x / y / z: M2 0.180 / 0.218 / 0.038 mm, M2 + NDO 0.187 / 0.222 / 0.069;
chirp: 0.238 / 0.417 / 0.029 vs 0.244 / 0.423 / 0.099; l₁ = 1 is worse again).

**Why: the right-hand panels.** d̂ is not tracking fast nonlinear effects; it is a slow drift
(x down to about −0.3 mm/s, z up to about 1.2 mm/s over 60 s, y small). It is the gap between where
M2's absolute equations say the tip should settle and where the simulator settles: a level
mismatch of d̂ / a ≈ 1.2 / 15.5 ≈ 0.08 mm in z. The Smith prediction already starts from the
measured tip, so it carries that level for free, and adding d̂ on top counts it twice.

**What happens in the controller (corrected 2026-11-09).** The delay-aware NDOSMC also beats the
λ = 4 SMC without observer, but the observer is not why. With the observer on, the code sets the
switching gain to η = Φe^(−l₁t) + ε/l₁, which tends to 1/0.1 = 10; with it off, η = ε = 1. An SMC at
λ = 4 with η = 10 and no observer matches the NDOSMC on all six tests (whole-run RMSE mm, η = 1 →
η = 10 → NDOSMC): scored circle 1.286 → 1.211 → 1.206, held-out 2.047 → 1.956 → 1.948, triangle
1.069 → 1.014 → 1.012, square 1.125 → 1.051 → 1.049, squiggle 0.361 → 0.311 → 0.308, waypoints
0.363 → 0.273 → 0.274. With l₁ = 0.1 the observer is a 10 s filter (d̂˙ = l₁ (d − d̂)) and in closed
loop its d̂ is about a fifth of the actual model residual (RMS ≈ 0.1 vs 0.6 mm/s,
`scripts/smc_signals.py`). So at these gains the observer neither improves the model's prediction
nor, measurably, the controller; the larger switching gain does the work.

## Why the one-mode models are off

Nikoletta's one-mode model moves in the right direction at the right moments but about 4× too far
and with no delay: its static gain is b/k = 7.7 mm/psi (no gravity stiffness in its derivation),
against 2.0 mm/psi in the simulator, and it has neither the 0.5 s dead time nor the 0.6 s pressure
lag. Adding the gravity stiffness brings the gain to 1.5 mm/psi (now 25 % too small) and cuts the
error by about 3×, but it still leads the simulator by about 1.1 s because it has no delay or lag.
M2, M3 and F were identified from the simulator's data with those dynamics in, so they follow it
to about 0.1 mm; M3's inertia term halves M2's error on y.

## Error table: every model, both excitations, every axis

Runs the simulator and all models on both standard excitations (random steps seed 31, chirp;
60 s each) and tabulates, per axis: free-run RMSE, largest |error|, bias = mean error (all mm),
and fit %. Then the 0.5 s-ahead prediction errors of §5 and the static gains (+1 psi step held 10 s from 4.5 psi). **Bold** = best in its
column (smallest RMSE / |max| / |bias|, largest fit, gain closest to the simulator); "–" = the
model has no output on that axis. Takes about a minute.

In [ ]:
from IPython.display import HTML, display

STATS = (("rmse_mm", "RMSE (mm)", lambda v: v, min), ("max_abs_mm", "max |e| (mm)", lambda v: v, min),
         ("bias_mm", "bias (mm)", abs, min), ("fit_pct", "fit (%)", lambda v: v, max))
FMT = {"rmse_mm": "{:.3f}", "max_abs_mm": "{:.2f}", "bias_mm": "{:+.3f}", "fit_pct": "{:.1f}"}
CSS = "<style>.mc td,.mc th{padding:3px 8px;border:1px solid #bbb;text-align:center}.mc td:first-child{text-align:left}</style>"

all_metrics = {}
for kind in ("random_steps", "chirp"):
    cmds = CM.excitation(kind, 60.0, seed=31)
    tr = CM.simulate_truth(cmds)
    all_metrics[kind] = {m: CM.metrics(tr, CM.SIMULATE[m](cmds - CM.BASE)) for m in CM.MODELS}
    all_metrics[kind]["prediction"] = CM.prediction_metrics(tr, cmds - CM.BASE)

def shown(key, v):
    """The value as printed, so models that tie on screen are bolded together."""
    return float(FMT[key].format(v))

def best(kind, ax, key, score, pick):
    vals = [score(shown(key, all_metrics[kind][m][ax][key])) for m in CM.MODELS if all_metrics[kind][m][ax]]
    return pick(vals)

html = CSS
for kind in all_metrics:
    head = "<tr><th rowspan=2>model</th>" + "".join(f"<th colspan=4>{a}</th>" for a in "xyz") + "</tr><tr>"
    head += "".join(f"<th>{lab}</th>" for _ in "xyz" for _, lab, _, _ in STATS) + "</tr>"
    rows = ""
    for m in CM.MODELS:
        rows += f"<tr><td>{CM.LABELS[m]}</td>"
        for ax in "xyz":
            v = all_metrics[kind][m][ax]
            for key, _, score, pick in STATS:
                if v is None:
                    rows += "<td>–</td>"
                    continue
                txt = FMT[key].format(v[key])
                rows += f"<td><b>{txt}</b></td>" if score(shown(key, v[key])) == best(kind, ax, key, score, pick) else f"<td>{txt}</td>"
        rows += "</tr>"
    html += f"<h4>Excitation: {kind}</h4><table class=mc>{head}{rows}</table>"

gains = CM.static_gains()
ghead = "<tr><th>static gain (mm/psi)</th><th>simulator</th>" + "".join(f"<th>{CM.LABELS[m]}</th>" for m in CM.MODELS) + "</tr>"
grows = ""
for name, g in gains.items():
    closest = min(CM.MODELS, key=lambda m: abs(g[m] - g["truth"]))
    grows += f"<tr><td>{name}</td><td>{g['truth']:+.2f}</td>" + "".join(
        (f"<td><b>{g[m]:+.2f} ({100 * g[m] / g['truth']:.0f} %)</b></td>" if m == closest
         else f"<td>{g[m]:+.2f} ({100 * g[m] / g['truth']:.0f} %)</td>")
        for m in CM.MODELS) + "</tr>"
phead = "<tr><th>predictor, 0.5 s ahead</th>" + "".join(f"<th>{k} {a}</th>" for k in all_metrics for a in "xyz") + "</tr>"
prows = ""
names = list(all_metrics["chirp"]["prediction"])
for name in names:
    prows += f"<tr><td>{name}</td>"
    for kind in all_metrics:
        for a in "xyz":
            v = all_metrics[kind]["prediction"][name][a]
            b = min(round(all_metrics[kind]["prediction"][n][a], 3) for n in names)
            prows += f"<td><b>{v:.3f}</b></td>" if round(v, 3) == b else f"<td>{v:.3f}</td>"
    prows += "</tr>"
html += f"<h4>Predicting the tip one dead time ahead from the measured tip, RMSE (mm)</h4><table class=mc>{phead}{prows}</table>"
html += f"<h4>Static gain: tip change 10 s after +1 psi (% of the simulator)</h4><table class=mc>{ghead}{grows}</table>"
display(HTML(html))